In [1]:
from llama_index.readers.web import SimpleWebPageReader
urls = ["https://paulgraham.com/worked.html",
        "https://paulgraham.com/startupideas.html",
        "https://paulgraham.com/avg.html"]
docs = SimpleWebPageReader(html_to_text=True).load_data(urls)

In [2]:
# 2. Chunk documents (simple fixed-size)
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

corpus = []
for doc in docs:
    corpus.extend(chunk_text(doc.text))

print(f"Total chunks: {len(corpus)}")

Total chunks: 105


In [3]:
import bm25s
# 3. Tokenize + Build BM25 index
corpus_tokens = bm25s.tokenize(corpus, stopwords="en")

retriever = bm25s.BM25(k1=1.5, b=0.75)
retriever.index(corpus_tokens)

# 4. Save index (reuse later)
retriever.save("bm25_index", corpus=corpus)
print("BM25 index built and saved.")

c:\Workspace\projects\Hybrid-Retrieval-RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Finding newlines for mmindex: 100%|██████████| 177k/177k [00:00<00:00, 51.6MB/s]

BM25 index built and saved.


In [4]:
# 5. Query it
def bm25_search(query: str, top_k: int = 5):
    query_tokens = bm25s.tokenize(query, stopwords="en")
    
    # Pass corpus here — bm25s will resolve indices to text automatically
    results, scores = retriever.retrieve(query_tokens, corpus=corpus, k=top_k)
    
    hits = []
    for i in range(results.shape[1]):
        hits.append({
            "text": results[0, i],       # now a string, not an index
            "score": float(scores[0, i])
        })
    return hits

# Test it
results = bm25_search("how to get startup ideas", top_k=3)
for r in results:
    print(f"Score: {r['score']:.3f} | {r['text'][:120]}...")

Score: 2.577 | ![](https://s.turbifycdn.com/aah/paulgraham/bel-7.gif)| ![](https://sep.turbifycdn.com/ca/Img/trans_1x1.gif)| [![](https...
Score: 2.113 | YC we call ideas that grow naturally out of the founders' own experiences "organic" startup ideas. The most successful s...
Score: 2.077 | than enterprise software. [4] This gets harder as you get older. While the space of ideas doesn't have dangerous local m...


In [5]:
from sentence_transformers import SentenceTransformer

# Free model — good balance of speed vs quality
# 384 dimensions, very fast on CPU
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Embedding dimension: {model.get_embedding_dimension()}")

# Embed all chunks (same corpus from Phase 3)
print(f"Embedding {len(corpus)} chunks...")
embeddings = model.encode(
    corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")
# → (num_chunks, 384)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5988.37it/s]


Embedding dimension: 384
Embedding 105 chunks...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.22s/it]

Embeddings shape: (105, 384)


In [7]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(host="localhost", port=6333)

COLLECTION_NAME = "pg_essays"
VECTOR_DIM = model.get_embedding_dimension()  # 384

# Delete if exists, then create fresh
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_DIM,
        distance=Distance.COSINE
    )
)

# Upload vectors with payload (text stored alongside vector)
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={"text": corpus[i], "chunk_id": i}
    )
    for i in range(len(corpus))
]

client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"Indexed {len(points)} chunks into Qdrant ✅")

Indexed 105 chunks into Qdrant ✅


In [8]:
from qdrant_client.models import QueryRequest

def dense_search(query: str, top_k: int = 5):
    query_vector = model.encode(query).tolist()
    
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True
    )
    
    return [
        {
            "text": r.payload["text"],
            "score": r.score,
            "id": r.id
        }
        for r in results.points      # note: .points to unpack
    ]

In [9]:
query = "how to get startup ideas"

print("=" * 60)
print("BM25 (Keyword) Results")
print("=" * 60)
for r in bm25_search(query, top_k=3):
    print(f"Score: {r['score']:.3f} | {r['text'][:120]}...\n")

print("=" * 60)
print("Dense (Semantic) Results")
print("=" * 60)
for r in dense_search(query, top_k=3):
    print(f"Score: {r['score']:.3f} | {r['text'][:120]}...\n")
    print(r['text'])

BM25 (Keyword) Results


Score: 2.577 | ![](https://s.turbifycdn.com/aah/paulgraham/bel-7.gif)| ![](https://sep.turbifycdn.com/ca/Img/trans_1x1.gif)| [![](https...

Score: 2.113 | YC we call ideas that grow naturally out of the founders' own experiences "organic" startup ideas. The most successful s...

Score: 2.077 | than enterprise software. [4] This gets harder as you get older. While the space of ideas doesn't have dangerous local m...

Dense (Semantic) Results


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.73it/s]

Score: 0.673 | just starting out can't expect to excavate that much volume. So you have two choices about the shape of hole you start w...

just starting out can't expect to excavate that much volume. So you have two choices about the shape of hole you start with. You can either dig a hole that's broad but shallow, or one that's narrow and deep, like a well. Made-up startup ideas are usually of the first type. Lots of people are mildly interested in a social network for pet owners. Nearly all good startup ideas are of the second type. Microsoft was a well when they made Altair Basic. There were only a couple thousand Altair owners, but without this software they were programming in machine language. Thirty years later Facebook had the same shape. Their first site was exclusively for Harvard students, of which there are only a few thousand, but those few thousand users wanted it a lot. When you have an idea for a startup, ask yourself: who wants this right now? Who wants this so much th

In [10]:
from collections import defaultdict

def reciprocal_rank_fusion(
    bm25_results: list,
    dense_results: list,
    k: int = 60,
    top_k: int = 5
):
    scores = defaultdict(float)
    texts = {}

    # Score BM25 results by rank
    for rank, result in enumerate(bm25_results):
        text = result["text"]
        scores[text] += 1 / (k + rank + 1)
        texts[text] = result

    # Score Dense results by rank
    for rank, result in enumerate(dense_results):
        text = result["text"]
        scores[text] += 1 / (k + rank + 1)
        texts[text] = result

    # Sort by fused RRF score
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [
        {
            "text": text,
            "rrf_score": round(score, 6),
        }
        for text, score in ranked[:top_k]
    ]

In [11]:
def hybrid_search(query: str, top_k: int = 5, fetch_k: int = 20):
    # Fetch more candidates than needed before fusion
    # More candidates = better fusion signal
    bm25_results = bm25_search(query, top_k=fetch_k)
    dense_results = dense_search(query, top_k=fetch_k)

    fused = reciprocal_rank_fusion(
        bm25_results,
        dense_results,
        k=60,
        top_k=top_k
    )
    return fused

In [ ]:
query = "how to get startup ideas"

print("=" * 60)
print("BM25 Results")
print("=" * 60)
for r in bm25_search(query, top_k=3):
    print(f"Score: {r['score']:.4f} | {r['text'][:150]}...\n")

print("=" * 60)
print("Dense Results")
print("=" * 60)
for r in dense_search(query, top_k=3):
    print(f"Score: {r['score']:.4f} | {r['text'][:150]}...\n")

print("=" * 60)
print("Hybrid (RRF) Results")
print("=" * 60)
for r in hybrid_search(query, top_k=3):
    print(f"RRF Score: {r['rrf_score']} | {r['text'][:150]}...\n")

BM25 Results


Score: 2.1958 | it both descriptively and prescriptively, and it was the second part that scared me. I wanted YC to be good, so if how hard I worked set the upper bou...

Score: 2.0221 | write more there. I remember taking the boys to the coast on a sunny day in 2015 and figuring out how to deal with some problem involving continuation...

Score: 1.9788 | write essays, and work on YC. As YC grew, and I grew more excited about it, it started to take up a lot more than a third of my attention. But for the...

Dense Results


Batches: 100%|██████████| 1/1 [00:00<00:00, 114.95it/s]


Score: 0.3902 | as you could by writing software, of course, but I thought if you were really industrious and lived really cheaply, it had to be possible to make enou...

Score: 0.3860 | rockets would fly, and a word processor that my father used to write at least one book. There was only room in memory for about 2 pages of text, so he...

Score: 0.3721 | yourself drawn to some kind of work despite its current lack of prestige, it's a sign both that there's something real to be discovered there, and tha...

Hybrid (RRF) Results


Batches: 100%|██████████| 1/1 [00:00<00:00, 142.63it/s]

RRF Score: 0.031025 | write essays, and work on YC. As YC grew, and I grew more excited about it, it started to take up a lot more than a third of my attention. But for the...

RRF Score: 0.029514 | day life pretty constraining, just as someone from the present would if they were sent back 50 years in a time machine. When something annoys you, it ...

RRF Score: 0.029462 | write more there. I remember taking the boys to the coast on a sunny day in 2015 and figuring out how to deal with some problem involving continuation...



In [13]:
from sentence_transformers import CrossEncoder

# Free HuggingFace model — trained on MS MARCO (same dataset from Phase 2)
# Lightweight, runs fine on CPU
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512
)

c:\Workspace\projects\Hybrid-Retrieval-RAG\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Harsh\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7901.00it/s]


In [14]:
def rerank(query: str, candidates: list, top_k: int = 5):
    if not candidates:
        return []

    # Cross-encoder expects list of (query, doc) pairs
    pairs = [(query, c["text"]) for c in candidates]

    # Returns a score per pair — higher = more relevant
    scores = reranker.predict(pairs)

    # Attach scores back to candidates
    for i, candidate in enumerate(candidates):
        candidate["rerank_score"] = float(scores[i])

    # Sort by re-rank score
    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    return reranked[:top_k]

In [15]:
def hybrid_reranked_search(query: str, top_k: int = 5, fetch_k: int = 20):
    
    # Stage 1 — Hybrid retrieval (fast, wide net)
    candidates = hybrid_search(query, top_k=fetch_k, fetch_k=fetch_k)
    
    # Stage 2 — Re-rank (slow, precise)
    final = rerank(query, candidates, top_k=top_k)
    
    return final

In [36]:
query = "how do startup ideas come about"

# print("=" * 60)
# print("BM25 Only")
# print("=" * 60)
# for r in bm25_search(query, top_k=3):
#     print(f"  {r['text'][:150]}...\n")

# print("=" * 60)
# print("Dense Only")
# print("=" * 60)
# for r in dense_search(query, top_k=3):
#     print(f"  {r['text'][:150]}...\n")

# print("=" * 60)
# print("Hybrid (RRF)")
# print("=" * 60)
# for r in hybrid_search(query, top_k=3):
#     print(f"  {r['text'][:150]}...\n")

print("=" * 60)
print("Hybrid + Re-ranked  ← final ranking")
print("=" * 60)
context = ""
for r in hybrid_reranked_search(query, top_k=5):
    print(f"  Rerank Score: {r['rerank_score']:.4f}")
    # print(f"  {r['text']}...\n")
    context = f"  {r['text']}...\n"

Hybrid + Re-ranked  ← final ranking


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]

  Rerank Score: 6.0406
  Rerank Score: 5.7350
  Rerank Score: 5.3098
  Rerank Score: 5.3030
  Rerank Score: 5.1186
